In [43]:
import pandas as pd
import numpy as np
import scipy.stats as sci
import datetime

from pathlib import Path

In [44]:
df = pd.read_csv("../data/west_states_filtered.csv")

C:\Users\sbzav\AppData\Local\Temp\ipykernel_3496\925793815.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/west_states_filtered.csv")


In [33]:
# Confirming columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 664409 entries, 0 to 664408
Data columns (total 25 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Unnamed: 0       664409 non-null  int64  
 1   VAERS_ID         664409 non-null  int64  
 2   symptom          664409 non-null  object 
 3   RECVDATE         664409 non-null  object 
 4   STATE            664409 non-null  object 
 5   AGE_YRS          636556 non-null  float64
 6   SEX              664409 non-null  object 
 7   RPT_DATE         431 non-null     object 
 8   DIED             664409 non-null  object 
 9   HOSPITAL         664409 non-null  object 
 10  HOSPDAYS         35326 non-null   float64
 11  DISABLE          664409 non-null  object 
 12  RECOVD           664409 non-null  object 
 13  VAX_DATE         650653 non-null  object 
 14  ONSET_DATE       630935 non-null  object 
 15  NUMDAYS          613848 non-null  float64
 16  V_ADMINBY        664409 non-null  obje

In [34]:
## Removes unused columns due to Streamlit resource limitations
df.drop(columns=['Unnamed: 0','FORM_VERS','V_FUNDBY','V_ADMINBY','NUMDAYS','V_ADMINBY','V_FUNDBY','FORM_VERS','TODAYS_DATE','VAX_LOT','ONSET_YEAR','ONSET_MONTHYEAR','ONSET_DATE','VAX_DATE','RECOVD','DISABLE','HOSPDAYS','RPT_DATE'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 664409 entries, 0 to 664408
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   VAERS_ID         664409 non-null  int64  
 1   symptom          664409 non-null  object 
 2   RECVDATE         664409 non-null  object 
 3   STATE            664409 non-null  object 
 4   AGE_YRS          636556 non-null  float64
 5   SEX              664409 non-null  object 
 6   DIED             664409 non-null  object 
 7   HOSPITAL         664409 non-null  object 
 8   VAX_MANU         664409 non-null  object 
 9   VAX_DOSE_SERIES  583601 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 50.7+ MB


In [35]:
# (Extra) Dropping people without an AGE_YRS
df.dropna(axis=0, subset=['AGE_YRS'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 636556 entries, 0 to 664408
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   VAERS_ID         636556 non-null  int64  
 1   symptom          636556 non-null  object 
 2   RECVDATE         636556 non-null  object 
 3   STATE            636556 non-null  object 
 4   AGE_YRS          636556 non-null  float64
 5   SEX              636556 non-null  object 
 6   DIED             636556 non-null  object 
 7   HOSPITAL         636556 non-null  object 
 8   VAX_MANU         636556 non-null  object 
 9   VAX_DOSE_SERIES  561303 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 53.4+ MB


In [36]:
df["STATE"].unique()

array(['AZ', 'NM', 'AK', 'CA', 'CO', 'WY', 'WA', 'ID', 'MT', 'NV', 'OR',
       'UT'], dtype=object)

In [37]:
# (Extra) Dropping people without an VAX_DOSE_SERIES
df.dropna(axis=0, subset=['VAX_DOSE_SERIES'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 561303 entries, 0 to 664408
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   VAERS_ID         561303 non-null  int64  
 1   symptom          561303 non-null  object 
 2   RECVDATE         561303 non-null  object 
 3   STATE            561303 non-null  object 
 4   AGE_YRS          561303 non-null  float64
 5   SEX              561303 non-null  object 
 6   DIED             561303 non-null  object 
 7   HOSPITAL         561303 non-null  object 
 8   VAX_MANU         561303 non-null  object 
 9   VAX_DOSE_SERIES  561303 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 47.1+ MB


In [40]:
# (Extra) Only keeps west coast states
df_coastal = df[df["STATE"].isin(["CA","WA","OR"])]
df_coastal.info()

<class 'pandas.core.frame.DataFrame'>
Index: 365621 entries, 4 to 664408
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   VAERS_ID         365621 non-null  int64  
 1   symptom          365621 non-null  object 
 2   RECVDATE         365621 non-null  object 
 3   STATE            365621 non-null  object 
 4   AGE_YRS          365621 non-null  float64
 5   SEX              365621 non-null  object 
 6   DIED             365621 non-null  object 
 7   HOSPITAL         365621 non-null  object 
 8   VAX_MANU         365621 non-null  object 
 9   VAX_DOSE_SERIES  365621 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 30.7+ MB


In [42]:
df_coastal["STATE"].value_counts()

STATE
CA    258554
WA     67901
OR     39166
Name: count, dtype: int64

In [ ]:
# Replaces west_states_filtered with reduced dataset
# filepath = Path("../data/west_states_filtered_v2.csv")
# filepath.parent.mkdir(parents=True, exist_ok=True)
# df_coastal.to_csv(filepath, index=False)